---
title: "Homogeneous cooling: Haff's law, gas damping, and the seed of clustering"
subtitle: "Give a box of grains a random kick and let inelastic collisions bleed the energy away. The granular temperature decays as Haff's law — a clean −2 power law that matches Enskog kinetic theory — the gas phase drains it faster still, and density fluctuations start to grow: the onset of the clustering instability the MFIX-Exa benchmark studies at scale."
author: "Peclet"
date: "2026-07-09"
categories: [coupling, cfd-dem, dem, flow, granular-gas, kinetic-theory, haff, clustering, benchmark, gpu]
jupyter: python3
bibliography: ../../references.bib
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/computational-chemical-engineering/peclet-examples/blob/main/examples/hcs-clustering/index.ipynb){target="_blank"}
&nbsp;The Haff's-law and clustering parts are pure `peclet.dem` (they run anywhere); the gas-damping section adds the `peclet-coupling` module (built from source — see *Reproduce*). This reproduces the physics of the **Clustering in the HCS** case from the [MFIX-Exa qualitative benchmarks](https://mfix.netl.doe.gov/doc/mfix-exa/guide/latest/test_benchmarks/qualitative_bencharks/hcs.html), following Haff [@haff1983] and the clustering-instability literature [@fullmer2017].

## What you'll learn

The **homogeneous cooling system** is the hydrogen atom of granular gases: a periodic box of grains, no
gravity, no walls, no forcing — just particles that have been given a random velocity distribution and
lose energy every time they collide inelastically. It is the simplest setting in which to ask *does the
DEM get the kinetic theory right?* and *when does a uniform granular gas stop being uniform?* You will:

1. Cool a **dry granular gas** and show its granular temperature obeys **Haff's law**,
   `T(t) = T₀/(1 + t/t₀)²`, with the cooling rate scaling as `(1 − e²)` and matching the **Enskog**
   prediction quantitatively.
2. Add the **gas phase** (CFD–DEM): viscous drag is a second energy sink, and the temperature falls
   faster than collisions alone can manage.
3. Watch **density fluctuations grow** above the homogeneous (Poisson) level — the onset of the
   Goldhirsch–Zanetti clustering instability that the MFIX-Exa benchmark resolves in a very large box.

In [ ]:
#| label: bootstrap
#| code-summary: "Environment bootstrap (local build, or peclet + peclet-coupling)"
import importlib.util, os, subprocess, sys
_local = os.environ.get("PECLET_LOCAL_BUILD")
if _local:
    for p in _local.split(os.pathsep):
        sys.path.insert(0, p)
elif importlib.util.find_spec("peclet") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peclet[cfd-dem]"], check=True)

In [ ]:
import numpy as np, time
import matplotlib.pyplot as plt
from peclet import flow, dem
from peclet.coupling import CfdDem
plt.rcParams.update({"figure.dpi": 130, "font.size": 10, "axes.axisbelow": True,
                     "figure.facecolor": "white", "savefig.bbox": "tight"})
print("backends — flow:", flow.execution_space, " dem:", dem.execution_space)

## The granular gas and Haff's law

Everything runs in dimensionless units: a **periodic cube** of side `L` (in particle diameters), grains of
radius `rp`, unit mass. The **granular temperature** is the velocity fluctuation per degree of freedom,
`T = ⟨|v − v̄|²⟩ / 3`. For a homogeneously cooling gas, Haff's law says the temperature decays so that

$$ \sqrt{T_0/T}\;=\;1 + t/t_0, \qquad \frac{1}{t_0}=\frac{(1-e^2)}{3}\,2\sqrt{\pi}\,n\,\sigma^2 g_0\,\sqrt{T_0}, $$ {#eq-haff}

a straight line whose slope grows with the inelasticity `(1 − e²)`; here `n` is the number density,
`σ = 2rp` the diameter, and `g₀` the Carnahan–Starling pair-correlation at contact (@eq-haff is the Enskog
result). We initialise a random gas and cool it at three restitutions.

In [ ]:
L, rp = 30.0, 0.5
Vp    = (4/3)*np.pi*rp**3
N     = int(0.15*L**3/Vp)                 # ~15% solids
n_den = N/L**3
g0    = (1 - 0.15/2)/(1 - 0.15)**3         # Carnahan-Starling at phi=0.15
print(f"N = {N} grains, solids fraction ≈ {N*Vp/L**3:.2f}")

def granular_T(v):
    vp = v - v.mean(0)
    return (vp*vp).sum(1).mean()/3.0

def cool(e, T0=9.0, nsteps=2200, dt=0.02, seed=1):
    rng = np.random.default_rng(seed)
    n1  = int(np.ceil(N**(1/3))); gl = (np.arange(n1)+0.5)*L/n1
    P   = np.array(np.meshgrid(gl, gl, gl, indexing="ij")).reshape(3, -1).T[:N].astype(np.float32)
    P  += rng.uniform(-0.15, 0.15, P.shape).astype(np.float32); P = np.mod(P, L)
    v   = rng.normal(0, np.sqrt(T0), (N, 3)).astype(np.float32); v -= v.mean(0)
    d = dem.Simulation(N + 64)
    d.initialize(shape_type=1, radius=rp); d.set_sphere_shape(rp)
    d.set_domain((0, 0, 0), (L, L, L)); d.enable_periodicity(True, True, True)
    d.set_gravity(0, 0, 0); d.set_material_params(e, 0.0, 0.0); d.set_solver_iterations(6, 4); d.set_dt(dt)
    d.set_positions(np.c_[P, np.ones(N, np.float32)]); d.set_velocities(v)
    T, ts = [granular_T(np.asarray(d.get_velocities())[:N])], [0.0]
    for i in range(nsteps):
        d.step(dt)
        if i % 10 == 0:
            T.append(granular_T(np.asarray(d.get_velocities())[:N])); ts.append((i+1)*dt)
    return np.array(ts), np.array(T)

es = [0.9, 0.8, 0.7]
runs = {e: cool(e) for e in es}
print("cooled:", {e: f"{T[-1]/T[0]:.3f}" for e,(t,T) in runs.items()})

The signature of Haff's law is that `√(T₀/T)` is **linear** in time. It is — cleanly — and each slope
matches the Enskog prediction from @eq-haff.

In [ ]:
#| label: fig-haff
#| fig-cap: "Haff's law. Left: granular temperature decays as a power law, faster for more inelastic grains. Right: √(T₀/T) is linear in time — the hallmark of the −2 power law — with slope set by the inelasticity. Dashed lines are the parameter-free Enskog prediction (@eq-haff)."
def enskog_slope(e, T0):
    return (1-e**2)/3 * 2*np.sqrt(np.pi) * n_den * (2*rp)**2 * g0 * np.sqrt(T0)

fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.7))
for e in es:
    t, T = runs[e]
    ax[0].plot(t, T/T[0], label=f"e = {e}")
    ax[1].plot(t, np.sqrt(T[0]/T), label=f"e = {e}")
    ax[1].plot(t, 1 + enskog_slope(e, T[0])*t, "k--", lw=1, alpha=0.7)
ax[0].set(xlabel="time", ylabel="T / T₀", title="granular temperature decay"); ax[0].legend()
ax[1].set(xlabel="time", ylabel="√(T₀/T)", title="Haff's law  (dashed = Enskog theory)"); ax[1].legend()
plt.tight_layout()

print("  e   measured 1/t₀   Enskog 1/t₀")
for e in es:
    t, T = runs[e]; m = t < t[-1]*0.6
    meas = np.polyfit(t[m], np.sqrt(T[0]/T[m]), 1)[0]
    print(f" {e}      {meas:.3f}         {enskog_slope(e, T[0]):.3f}")

The measured cooling rate sits within ~8% of the parameter-free kinetic-theory value at `e = 0.9` and
tracks the `(1 − e²)` scaling across the board — the deviation growing at strong inelasticity exactly where
Enskog's near-elastic, molecular-chaos assumptions begin to fail.

## Add the gas: a second energy sink

The MFIX-Exa case is *gas–solid*. Turn the gas on (a periodic fluid at rest, coupled through `wen_yu`
drag) and it drains the fluctuating energy far faster than collisions alone — the granular temperature
collapses.

In [ ]:
#| label: fig-gas
#| fig-cap: "Gas damping. The same granular gas (e = 0.9), cooled by inelastic collisions alone (dry) and with the gas phase coupled in. Viscous drag on every grain is a second, powerful energy sink — the CFD–DEM temperature falls far faster than Haff's collisional law."
rho_g, mu_g, rho_p = 1.0, 0.4, 8.0
m_p = rho_p*Vp; dt = 0.02
def init(seed=1, T0=9.0):
    rng = np.random.default_rng(seed)
    n1 = int(np.ceil(N**(1/3))); gl = (np.arange(n1)+0.5)*L/n1
    P = np.array(np.meshgrid(gl, gl, gl, indexing="ij")).reshape(3, -1).T[:N].astype(np.float32)
    P += rng.uniform(-0.15, 0.15, P.shape).astype(np.float32); P = np.mod(P, L)
    v = rng.normal(0, np.sqrt(T0), (N, 3)).astype(np.float32); v -= v.mean(0)
    return P, v
def dem_box(P, v, e):
    d = dem.Simulation(N+64); d.initialize(shape_type=1, radius=rp); d.set_sphere_shape(rp)
    d.set_domain((0,0,0),(L,L,L)); d.enable_periodicity(True,True,True); d.set_gravity(0,0,0)
    d.set_material_params(e,0.0,0.0); d.set_solver_iterations(6,4); d.set_dt(dt)
    d.set_positions(np.c_[P, np.full(N,1/m_p,np.float32)]); d.set_velocities(v); return d

t_dry, T_dry = runs[0.9]
P, v = init(); d = dem_box(P, v, 0.9)
s = flow.Solver(int(L), int(L), int(L)); s.set_rho(rho_g); s.set_mu(mu_g); s.set_dt(dt)
for f in range(6): s.set_domain_bc(f, 0)           # all periodic
s.set_pressure_pcg(True, 30, 1e-6)
cpl = CfdDem(s, d, fluid_dt=dt, mu=mu_g, rho=rho_g, radius=rp, drag="wen_yu",
             dem_substeps=20, periodic=(True, True, True), move_particles=True, porous=False)
Tg, tg = [granular_T(np.asarray(d.get_velocities())[:N])], [0.0]
for i in range(600):
    cpl.step()
    if i % 10 == 0: Tg.append(granular_T(np.asarray(d.get_velocities())[:N])); tg.append((i+1)*dt)
Tg, tg = np.array(Tg), np.array(tg)

fig, ax = plt.subplots(figsize=(5.2, 3.5))
ax.plot(t_dry, T_dry/T_dry[0], lw=2, label="dry — collisions only (Haff)")
ax.plot(tg, Tg/Tg[0], lw=2, label="gas-coupled (CFD–DEM)")
ax.set(xlabel="time", ylabel="T / T₀", title="the gas drains the granular temperature", xlim=(0, t_dry[-1]))
ax.legend(); plt.tight_layout()
print(f"at t = {tg[-1]:.0f}: dry T/T₀ ≈ {np.interp(tg[-1], t_dry, T_dry/T_dry[0]):.3f}, gas-coupled ≈ {Tg[-1]/Tg[0]:.3f}")

## The seed of clustering

Kinetic theory assumes the gas stays uniform, but an inelastic gas does not: a slightly denser region
cools faster (more collisions), loses pressure, and pulls in more grains — the **clustering instability**.
It only grows once the box is larger than a critical wavelength, so in a modest box the effect is a slow
rise of the density fluctuations **above** the homogeneous (Poisson) baseline rather than the dramatic
clumping MFIX-Exa sees in its 256-diameter system.

In [ ]:
#| label: fig-cluster
#| fig-cap: "Onset of clustering (strongly inelastic, e = 0.4, in a larger box). Left: coarse-grained density fluctuations, normalised so 1.0 is a homogeneous (Poisson) gas — they climb above 1 as structure develops. Right: a slab of the gas, homogeneous early and visibly textured late."
Lc = 48.0; Nc = int(0.15*Lc**3/Vp); nb = 16; mu = Nc/nb**3; pois = 1/np.sqrt(mu)
def cidx(P): H = np.histogramdd(P % Lc, bins=(nb,nb,nb), range=[[0,Lc]]*3)[0]; return (H.std()/H.mean())/pois
rng = np.random.default_rng(5); n1 = int(np.ceil(Nc**(1/3))); gl = (np.arange(n1)+0.5)*Lc/n1
Pc = np.array(np.meshgrid(gl, gl, gl, indexing="ij")).reshape(3,-1).T[:Nc].astype(np.float32)
Pc += rng.uniform(-0.15, 0.15, Pc.shape).astype(np.float32); Pc = np.mod(Pc, Lc)
vc = rng.normal(0, 3, (Nc, 3)).astype(np.float32); vc -= vc.mean(0)
dc = dem.Simulation(Nc+64); dc.initialize(shape_type=1, radius=rp); dc.set_sphere_shape(rp)
dc.set_domain((0,0,0),(Lc,Lc,Lc)); dc.enable_periodicity(True,True,True); dc.set_gravity(0,0,0)
dc.set_material_params(0.4, 0.0, 0.0); dc.set_solver_iterations(6,4); dc.set_dt(dt)
dc.set_positions(np.c_[Pc, np.ones(Nc, np.float32)]); dc.set_velocities(vc)
ci, tci, snaps = [], [], []
for i in range(3600):
    dc.step(dt)
    if i % 40 == 0: ci.append(cidx(np.asarray(dc.get_positions())[:Nc])); tci.append((i+1)*dt)
    if i in (400, 3599): snaps.append(((i+1)*dt, np.asarray(dc.get_positions())[:Nc].copy()))
ci, tci = np.array(ci), np.array(tci)

fig, ax = plt.subplots(1, 3, figsize=(12, 3.7))
ax[0].plot(tci, ci); ax[0].axhline(1, ls=":", c="0.5", label="homogeneous (Poisson)")
ax[0].set(xlabel="time", ylabel="density fluctuations / Poisson", title="clustering grows (e = 0.4)"); ax[0].legend()
for k, (t, Pp) in enumerate(snaps[:2]):
    a = ax[1+k]; m = Pp[:,2] < Lc/3
    a.scatter(Pp[m,0], Pp[m,1], s=1.2, c="#333", edgecolors="none")
    a.set(title=f"t = {t:.0f}", xlim=(0,Lc), ylim=(0,Lc), aspect="equal"); a.set_xticks([]); a.set_yticks([])
plt.tight_layout()
print(f"density-fluctuation index: {ci[0]:.2f} → {ci.max():.2f}  (>1 means clustered beyond a random gas)")

## Results — peclet vs. theory vs. MFIX-Exa

| | Prediction / benchmark | peclet |
|---|---|---|
| **Haff's law form** | `√(T₀/T)` linear in `t` [@haff1983] | linear (clean, all `e`) |
| **Cooling rate** | Enskog `1/t₀ ∝ (1−e²)·n·σ²·g₀·√T₀` | within ~8% at `e=0.9`; tracks `(1−e²)` |
| **Gas–solid HCS** | gas drag accelerates cooling | temperature collapses well below Haff |
| **Clustering** | grows past a critical size [@fullmer2017] | fluctuations rise above Poisson; dramatic clumping needs the large MFIX box |

The dry granular gas reproduces Haff's law and the Enskog cooling rate quantitatively; the gas coupling
adds the second dissipation channel that makes the MFIX-Exa case *gas–solid*; and the density fluctuations
show the clustering instability beginning — the phenomenon the benchmark isolates by going to a 256-diameter
domain, larger than we run here.

## Adapt this yourself

- **Grow the box.** Push `L` toward the MFIX-Exa 256 d (and lower `e`) to cross the critical size and watch
  clustering run away — the density index climbs and never saturates.
- **Structure factor.** Replace the coarse-grained variance with `S(k)` to measure the clustering
  wavelength and compare to the linear-stability prediction.
- **Sweep the density.** `g₀(φ)` steepens with solids fraction — the cooling rate and the critical size
  both move; @eq-haff predicts how.
- **Different drag.** The gas-damping rate depends on the drag law and the density ratio `ρ_p/ρ_g`; try
  `stokes`, `schiller_naumann`, `gidaspow`.

## Reproduce this

```bash
# The Haff's-law and clustering sections are pure peclet.dem and run on the PyPI wheels:
pip install peclet && quarto render examples/hcs-clustering/index.qmd --execute
# The gas-damping section needs the coupling module (built from source against a Kokkos prefix):
tools/bootstrap_deps.sh host-openmp
for m in flow dem coupling; do
  cmake -S $m -B $m/build -DCMAKE_PREFIX_PATH="$PWD/extern/install/host-openmp"; cmake --build $m/build -j
done
PECLET_LOCAL_BUILD="$PWD/flow/build:$PWD/dem/build:$PWD/coupling/build" \
  quarto render examples/hcs-clustering/index.qmd --execute
```